In [6]:
from dotenv import load_dotenv
load_dotenv()

True

In [7]:
from openai import OpenAI
import os

openai_client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url="https://api.k2think.ai/v1"
)

In [8]:
def llm(prompt):
    response = openai_client.chat.completions.create(
        model="MBZUAI-IFM/K2-Think-v2",
        messages = [
            {
            "role":"user",
            "content":prompt
        }
        ]
    )
    return response.choices[0].message.content

In [9]:
from collections import namedtuple
def llm_apresparse(prompt):
    output = llm(prompt)
    output_parsed = output.split("</think>")
    response = namedtuple('response', ['thinking', 'answer'])
    return response(thinking = output_parsed[0], answer = output_parsed[1])

In [10]:
context = '''General Course-Related Questions
#I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

edit on GitHub
#Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

edit on GitHub
#What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs.

Students participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the announcements channel on Telegram &amp; Slack before it begins. You can also watch live on the DataTalksClub YouTube Channel.

Don’t post questions in chat as they may be missed if the room is very active.

edit on GitHub
#Cloud alternatives with GPU
Check the quota and reset cycle carefully. Is the free hours limit per month or per week? Usually, if you change the configuration, the free hours quota might also be adjusted, or it might be billed separately.

Potential options include:

Google Colab
Kaggle
Databricks (possibly)
Consider using GPTs to discover more options. Be aware that some platforms might have restrictions on what you can and cannot install, so ensure to read what is included in the free vs paid tier.

edit on GitHub
#Leaderboard: I am not on the leaderboard / how do I know which one I am on the leaderboard?
When you set up your account, you are automatically assigned a random name, such as “Lucid Elbakyan.” Click on the "Jump to your record on the leaderboard" link to find your entry.

If you want to see what your Display name is, click on the "Edit Course Profile" button.

image #1

First field: This is your nickname/displayed name. You can change it if you want to be known by your Slack username, GitHub username, or any other nickname of your choice. This is useful if you want to remain anonymous.
Second field: Change this to your official name as in your identification documents—passport, national ID card, driver's license, etc. This is mandatory if you do not want "Lucid Elbakyan" on your certificate. This name will appear on your Certificate!
edit on GitHub
#Certificate: Can I follow the course in a self-paced mode and get a certificate?
No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.

You can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.'''

In [11]:
question = '''I just got to know about the course, caan i still join'''
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [12]:
answer = llm_apresparse(prompt).answer
print(answer)


Yes, you can still join the course. However, if you want a certificate you must submit your project while the submission window is still open.



In [13]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [14]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [15]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    # if something is broken then raaise error (above)
    course_data = course_response.json()

    documents.extend(course_data)

print(len(documents))
documents[2]

1350


{'id': '3f1424af17',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

## Search

In [16]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"], #for ranking
    keyword_fields=["course"] #for mattching no matter
)

index.fit(documents)

In [17]:
# if we want to say that 1 field is more useful than another,
# then boosting...
search_result = index.search(
    question, 
    boost_dict={'question' : 2.0,  "section": 0.5}, #0.5 => less imp
    filter_dict={'course': 'llm-zoomcamp'}, 
    num_results=5)
search_result

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '9f689c185f',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I missed the first homework - can I still get a certificate?',
  'answer': 'Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learn

In [18]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course} # for keyword search

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )
#search_result = search(question) #the way to use it

## Building Prompt

In [19]:
'''

prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

'''

'\n\nprompt = f"""\nYour task is to answer questions from the course participants\nbased on the provided context.\n\nUse the context to find relevant information and provide accurate\nanswers. If the answer is not found in the context,\nrespond with "I don\'t know."\n\nQuestion:\n{question}\n\nContext:\n{context}\n"""\n\n'

In [20]:
INSTRUCTIONS = '''Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know.'''

In [21]:
USER_PROMPT_TEMPLATE = '''
Question:
{question}

Context:
{context}'''

In [22]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [23]:
def build_prompt(question, search_result):  
    context = build_context(search_result)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question, 
        context=context)
    return prompt.strip()

In [24]:
prompt = build_prompt(question, search_result)
# OR prompt = build_prompt(question, search(question))
print(prompt)

Question:
I just got to know about the course, caan i still join

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: I missed the first homework - can I still get a certificate?
A: Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-

## LLM (RAG Pipeline)

In [25]:
#previous code
'''
def llm(prompt):
    response = openai_client.chat.completions.create(
        model="MBZUAI-IFM/K2-Think-v2",
        messages = [
            {
            "role":"user",
            "content":prompt
        }
        ]
    )
    return response.choices[0].message.content
output = llm(prompt)
output_parsed = output.split("</think>")
x = response_f(thinking = output_parsed[0], answer = output_parsed[1])
print(x.answer)
'''

'\ndef llm(prompt):\n    response = openai_client.chat.completions.create(\n        model="MBZUAI-IFM/K2-Think-v2",\n        messages = [\n            {\n            "role":"user",\n            "content":prompt\n        }\n        ]\n    )\n    return response.choices[0].message.content\noutput = llm(prompt)\noutput_parsed = output.split("</think>")\nx = response_f(thinking = output_parsed[0], answer = output_parsed[1])\nprint(x.answer)\n'

In [26]:
response_f = namedtuple('response', ['thinking', 'answer'])
response = openai_client.chat.completions.create(
    model="MBZUAI-IFM/K2-Think-v2",
    messages = [
        {
        "role":"user",
        "content":prompt
    }
    ]
)
output = response.choices[0].message.content
print(response.model_dump_json(indent=2))


{
  "id": "chatcmpl-7f63bd54-bb33-4954-afa1-c6135a454f9b",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "The user says: \"Question: I just got to know about the course, caan i still join\". There's context: multiple Q&A pieces covering similar queries.\n\nWe need to answer the question, presumably referencing the Q&A about joining late: \"Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.\"\n\nWe need to produce a concise answer, possibly mention that the course materials are still available and you can join anytime, but certificate requires project submission within the acceptance window.\n\nPotentially we can also encourage them to watch videos, do notebooks, etc.\n\nThus answer: Yes you can join even after discovering. Materials are up; you can start; but to get certificate you need to submit capstone project before deadline; hom

In [27]:
response.usage

CompletionUsage(completion_tokens=762, prompt_tokens=643, total_tokens=1405, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=0, rejected_prediction_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=None, cached_tokens=0))

In [28]:
message_history = [
    {'role': 'system', 'content': INSTRUCTIONS},
    {"role": "user", "content": prompt}
]

response = openai_client.chat.completions.create(
    model="MBZUAI-IFM/K2-Think-v2",
    messages = message_history
)
response.choices[0].message.content

'The user asks: "Question: I just got to know about the course, caan i still join". This is a question about whether they can still join after discovering the course. The context includes a specific answer: under "General Course-Related Questions" -> "Q: I just discovered the course. Can I still join? A: Yes, but if you want to receive a certificate, you need to submit your project while we\'re still accepting submissions."\n\nThus the answer is yes, you can still join, but for certificate you need to submit project while submissions are still open.\n\nThe user spelled "caan", but that\'s fine. They just want to know if they can still join.\n\nThus answer: yes, you can still join. However to get a certificate you need to submit your project while they are still accepting submissions.\n\nProbably we can incorporate that and also maybe note that you can start learning irrespective of registration. But the direct answer is just the same as context.\n\nWe must answer based on context. So f

In [29]:
#modfying the llm() function to make it like this

def llm(instructions, user_prompt, model="MBZUAI-IFM/K2-Think-v2"):

    message_history = [
        {'role': 'system', 'content': instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.chat.completions.create(
        model=model,
        messages = message_history
    )
    return response.choices[0].message.content

In [30]:
# Final RAG
def rag(query, model="MBZUAI-IFM/K2-Think-v2"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [31]:
answer = rag(question)
print(answer)

The user asks: "I just got to know about the course, caan i still join". The context includes a direct Q&A about this scenario: "General Course-Related Questions Q: I just discovered the course. Can I still join? A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions." So the answer is "Yes, you can still join. However, to get a certificate, you need to submit your project while we're still accepting submissions."

We will respond accordingly.

We should use the context: Yes, you can still join. Provide note about certificate.

So final answer will be: Yes, you can still join. If you want a certificate, you need to submit your project while submissions are open.

Thus a concise answer.

Probably also we could mention that you can start whenever you want, check the deadlines or platform for submission windows.

Thus answer: Yes, you can still join. However, for a certificate you need to submit your project while the subm

In [32]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [33]:
response = openai_client.chat.completions.create(
        model="MBZUAI-IFM/K2-Think-v2",
        messages = message_history,
        tools=[search_tool]
    )

BadRequestError: Error code: 400 - {'detail': 'Invalid request. Please check your API parameters and try again. For more details, please refer to: https://ifm.ai/docs/quick-start', 'error': {'message': '1 validation error:\n  {\'type\': \'missing\', \'loc\': (\'body\', \'tools\', 0, \'function\'), \'msg\': \'Field required\', \'input\': {\'type\': \'function\', \'name\': \'search\', \'description\': \'Search the FAQ database for entries matching the given query.\', \'parameters\': {\'type\': \'object\', \'properties\': {\'query\': {\'type\': \'string\', \'description\': \'Search query text to look up in the course FAQ.\'}}, \'required\': [\'query\'], \'additionalProperties\': False}}}\n\n  File "/usr/local/lib/python3.12/dist-packages/vllm/entrypoints/utils.py", line 34, in create_chat_completion\n    POST /v1/chat/completions [{\'type\': \'missing\', \'loc\': (\'body\', \'tools\', 0, \'function\'), \'msg\': \'Field required\', \'input\': {\'type\': \'function\', \'name\': \'search\', 